Example notebook `python/Notebooks/snippets/visualization.ipynb`: the examples of the user manual on the renderer, the
visualization settings, the model view and the solution viewer (the page *Renderer, graphics and visualization*); each example is tagged `part-<name>` and shown where the manual explains it. Without a
window - as here - the renderer functions return without drawing.

In [1]:
import exudyn
import exudyn as exu
from exudyn.utilities import *
import numpy as np

SC = exu.SystemContainer()
mbs = SC.AddSystem()
oGround = mbs.CreateGround()
oMass = mbs.CreateMassPoint(referencePosition=[1,0,0], initialVelocity=[0,1,0], mass=1)
oSpring = mbs.CreateSpringDamper(itemNumbers=[oGround, oMass], stiffness=100, damping=1)
nMarker = mbs.AddMarker(MarkerBodyPosition(bodyNumber=oMass))
mbs.AddSensor(SensorBody(bodyNumber=oMass, storeInternal=True, outputVariableType=exu.OutputVariableType.Position))
mbs.variables['nTrackNode'] = mbs.GetObject(oMass)['nodeNumber']
mbs.Assemble()

## The dialog of the visualization settings

In [2]:
import exudyn.misc.GUI
exudyn.misc.GUI.dialogDefaultWidth             #unscaled width of, e.g., right-mouse-button dialog
exudyn.misc.GUI.treeEditDefaultWidth = 800
exudyn.misc.GUI.treeEditDefaultHeight = 600
exudyn.misc.GUI.treeEditMaxInitialHeight = 600 #otherwise height is increased for larger screens
exudyn.misc.GUI.treeEditOpenItems = ['general','contact'] #these tree items are opened when a dialog is opened the first time
exudyn.misc.GUI.treeEditLastOpenItems          #which folders were open when a dialog was last used in this process; the next dialog opens with them (None until a dialog was used)
#
exudyn.misc.GUI.treeviewDefaultFontSize        #this is the base font size of the dialog (also right-mouse-button dialog)
exudyn.misc.GUI.useRenderWindowDisplayScaling  #if True, the scaling will follow the current scaling of the render window; if False, it will use the `tkinter` internal scaling, which uses the main screen where the dialog is created (which won't scale well, if the window is moved to another screen).
#
exudyn.misc.GUI.textHeightFactor = 1.45        #this factor is used to increase height of lines in tree view as compared to font size

## Visualization settings in the script

In [3]:
SC.visualizationSettings.nodes.defaultSize = 0.001      #draw nodes very small

#change openGL parameters; current values can be obtained from SC.renderer.GetState()
#change zoom factor:
SC.visualizationSettings.openGL.advanced.initialZoom = 0.2
#set the center point of the scene (can be attached to moving object):
SC.visualizationSettings.openGL.advanced.initialCenterPoint = [0.192, -0.0039,-0.075]

#turn of auto-fit:
SC.visualizationSettings.general.autoFitScene = False

#change smoothness of a cylinder:
SC.visualizationSettings.general.cylinderTiling = 100

#make round objects flat:
SC.visualizationSettings.openGL.advanced.shadeModelSmooth = False

#turn on coloured plot, using y-component of displacements:
SC.visualizationSettings.contour.outputVariable = exu.OutputVariableType.Displacement
SC.visualizationSettings.contour.outputVariableComponent = 1 #0=x, 1=y, 2=z

## Commands during a simulation

The solver of the last run is `mbs.sys['dynamicSolver']`:

In [4]:
mbs.SolveDynamic()

True

In [5]:
x=5 #or change any other variable used in Python user functions
print(mbs) #print current mbs overview
print(mbs.GetSensorValues(0))
#adjust simulation end time, in long-run simulations:
mbs.sys['dynamicSolver'].it.endTime = 1
#adjust output behavior
mbs.sys['dynamicSolver'].output.verboseMode = 0

<systemData: 
  Number of nodes= 1
  Number of objects = 3
  Number of markers = 3
  Number of loads = 0
  Number of sensors = 1
  Number of ODE2 coordinates = 3
  Number of ODE1 coordinates = 0
  Number of AE coordinates   = 0
  Number of data coordinates   = 0

For details see mbs.systemData, mbs.sys and mbs.variables
>

[0.56467641 0.84272909 0.        ]


In [6]:
n=mbs.systemData.NumberOfObjects()
for i in range(n):
    d = mbs.GetObject(i)
    #if 'Joint' in d['objectType']:
    if 'activeConnector' in d:
        mbs.SetObjectParameter(i, 'activeConnector', False)

## The render state

In [7]:
SC.renderer.Start()
renderState = SC.renderer.GetState()
for (key, value) in renderState.items():
    print(key, '=', value.tolist() if isinstance(value, np.ndarray) else value)

viewEnabled = True
windowOpen = False
centerPoint = [0.0, 0.0, 0.0]
rotationCenterPoint = [0.0, 0.0, 0.0]
maxSceneSize = 1.0
zoom = 1.0
boundingBox = [array([-1., -1., -1.]), array([1., 1., 1.])]
currentWindowSize = [1024, 768]
currentWindowPosition = [-1, -1]
displayScaling = 1.0
modelRotation = [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
projectionMatrix = [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]
mouseCoordinates = [0.0, 0.0]
openGLcoordinates = [0.0, 0.0]
mouseSelectionMbsNumber = 0
mouseSelectionItemType = ItemType._None
mouseSelectionItemID = 0
mouseSelectionZdepth = 0.0
joystickPosition = [0.0, 0.0, 0.0]
joystickRotation = [0.0, 0.0, 0.0]
joystickAvailable = -1


## Storing the model view

In [8]:
SC.visualizationSettings.general.autoFitScene = False #prevent from autozoom
SC.renderer.Start()
SC.renderer.SetModelView(zoom=8.8,rotationVector=[-0.8120557,0.4727261,0.7176849],centerPoint=[1.562,-1.526,0])
#+++++++++++++++
#do simulation here
#+++++++++++++++
SC.renderer.Stop()

In [9]:
SC.visualizationSettings.general.autoFitScene = False #prevent from autozoom
SC.renderer.Start()
SC.renderer.RestoreSavedState() #the view of the previous run, if there is one
#+++++++++++++++
#do simulation here and adjust model view settings with mouse
#+++++++++++++++

#store model view for next run:
SC.renderer.Stop() #stores render state in exu.sys['renderState']

In [10]:
SC.visualizationSettings.general.autoFitScene = False #prevent from autozoom
SC.renderer.Start()
renderState={'centerPoint': [1.0, 0.0, 0.0],
             'maxSceneSize': 2.0,
             'zoom': 1.0,
             'modelRotation':    [[ 0.34202015,  0.        ,  0.9396926 ],
                                  [-0.60402274,  0.76604444,  0.21984631],
                                  [-0.7198463 , -0.6427876 ,  0.26200265]]}
SC.renderer.SetState(renderState)
#.... further code for simulation here
SC.renderer.Stop()

## Camera following objects

In [11]:
SC.visualizationSettings.view0.camera.trackMarker = nMarker

In [12]:
#mbs.variables['nTrackNode'] contains node number
def UFgraphics(mbs, objectNum):
    n = mbs.variables['nTrackNode']
    p = mbs.GetNodeOutput(n,exu.OutputVariableType.Position,
                          configuration=exu.ConfigurationType.Visualization)
    rs=SC.renderer.GetState() #get current render state
    A = np.array(rs['modelRotation'])
    p = A.T @ p #transform point into model view coordinates
    rs['centerPoint']=[p[0],p[1],p[2]]
    SC.renderer.SetState(rs)  #modify render state
    return []

#add object with graphics user function
oGround2 = mbs.AddObject(ObjectGround(visualization=
               VObjectGround(graphicsDataUserFunction=UFgraphics)))
#.... further code for simulation here

## The solution viewer

In [13]:
SC.visualizationSettings.view0.camera.trackMarker = -1
mbs.Assemble()
mbs.SolveDynamic()

True

In [14]:
#... mbs has been previously solved
mbs.SolutionViewer()

#a solution file of an earlier run; OutputFilePath gives where the solver wrote it,
#also if exudyn.config.outputDirectory is set
from exudyn.utilities import LoadSolutionFile, OutputFilePath
sol = LoadSolutionFile(OutputFilePath('solution/coordinatesSolution.txt'))
mbs.SolutionViewer(solution=sol)

columns imported = [3, 3, 3, 0, 0, 0, 0]
total columns to be imported = 9 , array size of file = 10
